In [ ]:
!pip install faiss-cpu sentence-transformers pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.9 MB/s eta 0:00:00


In [ ]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-04-23 08:11:12--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  4.32MB/s    in 0.2s    

2026-04-23 08:11:12 (4.32 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [ ]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
movies = pd.read_csv("ml-latest-small/movies.csv")
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
# Check missing values
print(movies.isnull().sum())

# Fill missing (if any)
movies['title'] = movies['title'].fillna("")
movies['genres'] = movies['genres'].fillna("")

# Clean genres (replace | with space)
movies['genres'] = movies['genres'].apply(lambda x: x.replace('|', ' '))

movieId    0
title      0
genres     0
dtype: int64


In [ ]:
movies['combined'] = movies['title'] + " " + movies['genres']
movies['combined'].head()

,combined
0,Toy Story (1995) Adventure Animation Children ...
1,Jumanji (1995) Adventure Children Fantasy
2,Grumpier Old Men (1995) Comedy Romance
3,Waiting to Exhale (1995) Comedy Drama Romance
4,Father of the Bride Part II (1995) Comedy


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(movies['combined'].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

Embedding shape: (9742, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total vectors in FAISS index:", index.ntotal)

Total vectors in FAISS index: 9742


In [ ]:
metadata = movies[['movieId', 'title', 'genres']].reset_index(drop=True)

In [ ]:
query = input("Enter your movie query: ")

Enter your movie query: a science fiction space adventure


In [ ]:
query_embedding = model.encode([query]).astype('float32')

In [ ]:
k = 5  # top results

distances, indices = index.search(query_embedding, k)

In [ ]:
print("\nTop Recommendations:\n")

for i, idx in enumerate(indices[0]):
    print(f"{i+1}. {metadata.iloc[idx]['title']} ({metadata.iloc[idx]['genres']})")
    print(f"   Similarity Score (distance): {distances[0][i]:.4f}\n")


Top Recommendations:

1. The Space Between Us (2016) (Adventure Sci-Fi)
   Similarity Score (distance): 0.4977

2. SpaceCamp (1986) (Adventure Sci-Fi)
   Similarity Score (distance): 0.6632

3. Journey to the Center of the Earth (2008) (Action Adventure Sci-Fi)
   Similarity Score (distance): 0.6664

4. Lost in Space (1998) (Action Adventure Sci-Fi)
   Similarity Score (distance): 0.6817

5. My Science Project (1985) (Adventure Sci-Fi)
   Similarity Score (distance): 0.6867



In [ ]:
def keyword_search(query, df, top_k=5):
    results = df[df['combined'].str.contains(query, case=False, na=False)]
    return results.head(top_k)

print("\n🔹 Keyword-Based Results:\n")
kw_results = keyword_search(query, movies)

for i, row in kw_results.iterrows():
    print(f"- {row['title']} ({row['genres']})")

print("\n🔹 Semantic (FAISS) Results:\n")
for i, idx in enumerate(indices[0]):
    print(f"- {metadata.iloc[idx]['title']} ({metadata.iloc[idx]['genres']})")


🔹 Keyword-Based Results:


🔹 Semantic (FAISS) Results:

- The Space Between Us (2016) (Adventure Sci-Fi)
- SpaceCamp (1986) (Adventure Sci-Fi)
- Journey to the Center of the Earth (2008) (Action Adventure Sci-Fi)
- Lost in Space (1998) (Action Adventure Sci-Fi)
- My Science Project (1985) (Adventure Sci-Fi)
